# 02 — Feature Engineering & PyTorch LSTM Training
**Predictive Equipment Failure — NASA CMAPSS FD001**

Runtime: **T4 GPU** (Runtime > Change runtime type > T4 GPU)

This notebook:
1. Installs dependencies & sets up the project on Colab
2. Runs the full feature engineering pipeline (RUL labels → windows → normalization)
3. Trains a dual-head PyTorch LSTM (RUL regression + failure classification)
4. Evaluates against NASA CMAPSS FD001 benchmarks
5. Exports `rul_predictor_v1.pt` and `scaler.joblib` to GCS

## Section 0 — Environment Setup

In [ ]:
!pip install -q torch numpy pandas scikit-learn matplotlib seaborn joblib tqdm google-cloud-storage

In [ ]:
import os, sys

# ── Detect environment ────────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Mount Google Drive (project files must be saved in Drive)
    from google.colab import drive
    drive.mount('/content/drive')

    # ── Update this path to where YOUR project lives in Drive ──────────────
    PROJECT_DIR = '/content/drive/MyDrive/predictive-maintenance-vertex'
    # ───────────────────────────────────────────────────────────────────────

    # Alternative: clone from GitHub once repo is pushed
    # import subprocess
    # subprocess.run(['git', 'clone',
    #     'https://github.com/DevMLAI01/Predictive-maintenance-MLOps.git',
    #     '/content/project'], check=True)
    # PROJECT_DIR = '/content/project'

    os.chdir(PROJECT_DIR)
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)
else:
    # Running locally: notebooks/ is one level below project root
    project_root = os.path.abspath(
        os.path.join(os.path.dirname(os.path.abspath('__file__')), '..')
    )
    # Fallback: assume cwd is already the project root
    if not os.path.isdir(os.path.join(project_root, 'src')):
        project_root = os.getcwd()
    os.chdir(project_root)
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

print(f'Working directory : {os.getcwd()}')
print(f'sys.path[0]       : {sys.path[0]}')
print(f'src/ exists       : {os.path.isdir("src")}')

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}  —  device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# Pull raw data from GCS (already uploaded in Phase 2 — no src import needed)
import os

GCS_BUCKET   = 'predictive-maintenance-artifacts'
RAW_DATA_DIR = 'data/raw'
os.makedirs(RAW_DATA_DIR, exist_ok=True)

files_needed = ['train_FD001.txt', 'test_FD001.txt', 'RUL_FD001.txt']
missing = [f for f in files_needed if not os.path.exists(f'{RAW_DATA_DIR}/{f}')]

if missing:
    print(f'Fetching {len(missing)} file(s) from GCS ...')
    # Authenticate (required in Colab; Cloud Shell is pre-authenticated)
    try:
        from google.colab import auth
        auth.authenticate_user()
    except ImportError:
        pass
    os.system(f'gsutil -m cp gs://{GCS_BUCKET}/data/raw/train_FD001.txt '
              f'gs://{GCS_BUCKET}/data/raw/test_FD001.txt '
              f'gs://{GCS_BUCKET}/data/raw/RUL_FD001.txt '
              f'{RAW_DATA_DIR}/')
    missing_after = [f for f in files_needed if not os.path.exists(f'{RAW_DATA_DIR}/{f}')]
    if missing_after:
        raise FileNotFoundError(
            f'GCS download failed for: {missing_after}\n'
            'Ensure you are authenticated and the bucket exists.'
        )
    print('Download complete.')
else:
    print(f'All raw files present in {RAW_DATA_DIR}/')

## Section 1 — Feature Engineering

Steps:
1. Load raw CSVs → DataFrames
2. Compute piecewise-linear RUL (capped at 125)
3. Drop 6 bad sensors (zero-variance + NaN correlation from EDA)
4. MinMaxScaler fit on train only
5. Create 30-cycle rolling windows → `(n_samples, 30, 15)` arrays

In [ ]:
from src.data.loader import CMAPSSLoader
from src.data.features import FeatureEngineer, SELECTED_SENSORS, DROP_SENSORS, N_FEATURES

loader = CMAPSSLoader(data_dir='data/raw')
train_df, test_df, rul_series = loader.load_all()

fe = FeatureEngineer(processed_dir='data/processed')

train_df = fe.compute_rul(train_df, rul_cap=125)
print(f'RUL range: {train_df["rul"].min():.0f} – {train_df["rul"].max():.0f} cycles')

print(f'\nDropping sensors: {DROP_SENSORS}')
train_df = fe.select_sensors(train_df)
test_df  = fe.select_sensors(test_df)
print(f'Retained sensors ({N_FEATURES}): {SELECTED_SENSORS}')

train_df, test_df, scaler = fe.normalize(train_df, test_df)

print('\nCreating rolling-window sequences ...')
X_train, y_train = fe.create_sequences(train_df, window_size=30)
X_test, y_test   = fe.create_sequences(test_df, window_size=30, is_test=True, rul_series=rul_series)

print(f'\nArray shapes:')
print(f'  X_train : {X_train.shape}   (n_samples, window, features)')
print(f'  y_train : {y_train.shape}')
print(f'  X_test  : {X_test.shape}')
print(f'  y_test  : {y_test.shape}')

fe.save_processed(X_train, y_train, X_test, y_test)

In [ ]:
from src.model.train import train_val_split_by_engine

X_tr, y_tr, X_val, y_val = train_val_split_by_engine(
    X_train, y_train, train_df, val_frac=0.2
)
print(f'Train   : {X_tr.shape[0]:,} sequences (engines 1–80)')
print(f'Val     : {X_val.shape[0]:,} sequences (engines 81–100)')

## Section 2 — Model Initialization

In [ ]:
from src.model.lstm import RULPredictor

model = RULPredictor(
    n_features=N_FEATURES,
    hidden_size=128,
    num_layers=2,
    dropout=0.2,
    window_size=30,
)

print(model)
print(f'\nTotal trainable parameters: {model.count_parameters():,}')
print(f'Expected ~450K params')

## Section 3 — Training

- Loss: MSELoss (RUL) + 0.3 × BCELoss (failure within 30 cycles)
- Optimizer: Adam, lr=0.001, weight_decay=1e-5
- Scheduler: ReduceLROnPlateau (patience=5, factor=0.5)
- Early stopping: patience=15 epochs

In [ ]:
import os
from src.model.train import Trainer

os.makedirs('model_artifacts', exist_ok=True)

config = {
    'learning_rate': 0.001,
    'batch_size': 256,
    'epochs': 100,
    'patience': 15,
    'device': 'cuda',
    'checkpoint_dir': 'model_artifacts',
    'experiment_name': 'rul-predictor-v1',
    'bce_weight': 0.3,
    'hidden_size': 128,
    'num_layers': 2,
    'window_size': 30,
    'rul_cap': 125,
    'n_features': N_FEATURES,
}

trainer = Trainer(model, config)
history = trainer.train(X_tr, y_tr, X_val, y_val)

In [ ]:
import matplotlib.pyplot as plt

epochs_  = [h['epoch']      for h in history]
tr_loss  = [h['train_loss'] for h in history]
val_loss = [h['val_loss']   for h in history]
val_rmse = [h['val_rmse']   for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs_, tr_loss,  label='Train loss')
ax1.plot(epochs_, val_loss, label='Val loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Training / Validation Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_, val_rmse, color='tomato', label='Val RMSE')
ax2.axhline(15.0, color='orange', linestyle='--', label='Good (15)')
ax2.axhline(13.0, color='green',  linestyle='--', label='Excellent (13)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('RMSE (cycles)'); ax2.set_title('Validation RMSE')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('model_artifacts/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Best val RMSE: {min(val_rmse):.2f} cycles')

## Section 4 — Evaluation on Held-Out Test Set

In [ ]:
import torch
import numpy as np
from src.model.evaluate import Evaluator

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.eval()

with torch.no_grad():
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    outputs  = model(X_test_t)
    y_pred   = outputs['rul'].squeeze(1).cpu().numpy()
    y_pred   = np.clip(y_pred, 0, None)

evaluator = Evaluator()
metrics   = evaluator.compute_metrics(y_test, y_pred)

evaluator.generate_report(metrics, save_path='model_artifacts/evaluation_report.json')
evaluator.plot_predictions(y_test, y_pred, save_path='model_artifacts/evaluation_plots')

## Section 5 — Export Artifact

In [ ]:
import joblib

trainer.save_model('model_artifacts/rul_predictor_v1.pt')
joblib.dump(scaler, 'model_artifacts/scaler.joblib')
print('Scaler saved → model_artifacts/scaler.joblib')
print(f"\nModel saved. RMSE: {metrics['rmse']:.2f} | NASA Score: {metrics['nasa_score']:.1f}")

## Section 6 — Upload Artifacts to GCS

Run these cells after confirming RMSE < 15 in the evaluation above.

In [ ]:
import os

GCS_BUCKET     = os.environ.get('GCS_BUCKET_NAME', 'predictive-maintenance-artifacts')
GCS_MODEL_PATH = f'gs://{GCS_BUCKET}/models/v1/'

# Authenticate if in Colab
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

print(f'Uploading to {GCS_MODEL_PATH} ...')
os.system(f'gsutil cp model_artifacts/rul_predictor_v1.pt    {GCS_MODEL_PATH}')
os.system(f'gsutil cp model_artifacts/scaler.joblib           {GCS_MODEL_PATH}')
os.system(f'gsutil cp model_artifacts/evaluation_report.json  {GCS_MODEL_PATH}')
print('Upload complete.')

# Verify
os.system(f'gsutil ls -lh {GCS_MODEL_PATH}')

---
## ✅ PHASE 3 COMPLETE

### Verification checklist
- [ ] `X_train` shape: `(n_samples, 30, 15)`
- [ ] `RULPredictor` forward pass: input `(32, 30, 15)` → output `{"rul": (32,1)}`
- [ ] Training completed without OOM on Colab T4
- [ ] Best checkpoint saved: `model_artifacts/best_model.pt`
- [ ] RMSE < 20 cycles on validation set (Phase 3 interim target)
- [ ] `evaluation_report.json` generated
- [ ] `rul_predictor_v1.pt` + `scaler.joblib` uploaded to `gs://predictive-maintenance-artifacts/models/v1/`

**Confirm to proceed to Phase 4 — Vertex AI Model Registration**